In [13]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings("ignore")

path_file = r'Student_Depression_Dataset.csv'
df = pd.read_csv(path_file)
df.head(10)



import torch
import torch.nn as nn
import torch.optim as optim

print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

df

GPU Available: True
GPU Name: NVIDIA GeForce RTX 3050


,Gender,Age,Family_background,Relationship_status,Marital_status,Living_style,Current_semester,CGPA,friend_circle,Health_condition,Cost_without_study,Home_division,Depression
0,Female,22,Middle class,Single,Unmarried,Bachelor,8,3.43,Yes,Critical,10113,Chittagong,Yes
1,Male,19,Middle class,Single,Unmarried,Bachelor,2,3.70,Yes,Abnormal,13815,Chittagong,Yes
2,Male,23,Middle class,Single,Unmarried,Family,8,3.28,No,Abnormal,15450,Dhaka,Yes
3,Male,20,Middle class,Complicated,Unmarried,Family,2,3.77,No,Good,17413,Dhaka,No
4,Female,21,Middle class,Single,Unmarried,Family,1,4.00,No,Abnormal,16693,Dhaka,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4459,Female,26,Middle class,In a Relationship,Separated,Family,11,2.97,No,Critical,12619,Khulna,Yes
4460,Male,28,Lower Middle class,Single,Unmarried,Family,8,2.60,No,Normal,2806,Rajshahi,Yes
4461,Female,23,Middle class,In a Relationship,Married,Bachelor,10,2.80,No,Good,14066,Rajshahi,No
4462,Female,23,Middle class,Complicated,Married,Family,5,3.00,Yes,Normal,14338,Barisal,No


In [14]:
print(df.shape)

print(df.dtypes)

print(df.isna().sum())

print(df["Depression"].value_counts(dropna=False))

print(df["Depression"].value_counts(normalize=True, dropna=False) * 100)

(4464, 13)
Gender                     str
Age                      int64
Family_background          str
Relationship_status        str
Marital_status             str
Living_style               str
Current_semester         int64
CGPA                   float64
friend_circle              str
Health_condition           str
Cost_without_study       int64
Home_division              str
Depression                 str
dtype: object
Gender                 0
Age                    0
Family_background      0
Relationship_status    0
Marital_status         0
Living_style           0
Current_semester       0
CGPA                   0
friend_circle          0
Health_condition       0
Cost_without_study     0
Home_division          0
Depression             0
dtype: int64
Depression
No     3263
Yes    1201
Name: count, dtype: int64
Depression
No     73.095878
Yes    26.904122
Name: proportion, dtype: float64


In [15]:
X = df.drop(columns=["Depression"])
y = df["Depression"].map({"No": 0, "Yes": 1})

In [16]:
print(X.shape)
print(y.value_counts())
print(y.head())

(4464, 12)
Depression
0    3263
1    1201
Name: count, dtype: int64
0    1
1    1
2    1
3    0
4    1
Name: Depression, dtype: int64


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [18]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nDistribuzione training:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribuzione validation:")
print(y_val.value_counts(normalize=True) * 100)

print("\nDistribuzione test:")
print(y_test.value_counts(normalize=True) * 100)

Training: (3124, 12)
Validation: (670, 12)
Test: (670, 12)

Distribuzione training:
Depression
0    73.111396
1    26.888604
Name: proportion, dtype: float64

Distribuzione validation:
Depression
0    73.134328
1    26.865672
Name: proportion, dtype: float64

Distribuzione test:
Depression
0    72.985075
1    27.014925
Name: proportion, dtype: float64


In [20]:
variabili_numeriche = [
    "Age",
    "Current_semester",
    "CGPA",
    "Cost_without_study"
]

variabili_categoriche = [
    "Gender",
    "Family_background",
    "Relationship_status",
    "Marital_status",
    "Living_style",
    "friend_circle",
    "Health_condition",
    "Home_division"
]

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessing = ColumnTransformer(
    transformers=[
        (
            "numeriche",
            StandardScaler(),
            variabili_numeriche
        ),
        (
            "categoriche",
            OneHotEncoder(handle_unknown="ignore"),
            variabili_categoriche
        )
    ]
)

In [22]:
X_train_array = preprocessing.fit_transform(X_train)

X_val_array = preprocessing.transform(X_val)

X_test_array = preprocessing.transform(X_test)

In [24]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo utilizzato:", device)


Dispositivo utilizzato: cuda


In [25]:
X_train_tensor = torch.tensor(
    X_train_array.toarray() if hasattr(X_train_array, "toarray") else X_train_array,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_array.toarray() if hasattr(X_val_array, "toarray") else X_val_array,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_array.toarray() if hasattr(X_test_array, "toarray") else X_test_array,
    dtype=torch.float32
)

In [26]:
y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

y_val_tensor = torch.tensor(
    y_val.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

In [27]:
print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)

print("X_val:", X_val_tensor.shape)
print("y_val:", y_val_tensor.shape)

print("X_test:", X_test_tensor.shape)
print("y_test:", y_test_tensor.shape)

X_train: torch.Size([3124, 31])
y_train: torch.Size([3124, 1])
X_val: torch.Size([670, 31])
y_val: torch.Size([670, 1])
X_test: torch.Size([670, 31])
y_test: torch.Size([670, 1])


In [28]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [29]:
class ReteNeurale(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.rete = nn.Sequential(

            nn.Linear(input_size, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)

        )

    def forward(self, x):

        return self.rete(x)

In [30]:
input_size = X_train_tensor.shape[1]

modello = ReteNeurale(input_size).to(device)

print(modello)

ReteNeurale(
  (rete): Sequential(
    (0): Linear(in_features=31, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [31]:
numero_parametri = sum(
    parametro.numel()
    for parametro in modello.parameters()
    if parametro.requires_grad
)

print("Numero di parametri:", numero_parametri)

Numero di parametri: 1569


In [32]:
criterio = nn.BCEWithLogitsLoss()


ottimizzatore = torch.optim.Adam(
    modello.parameters(),
    lr=0.001
)

In [33]:
numero_epoche = 100

for epoca in range(numero_epoche):

    modello.train()

    perdita_totale = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        ottimizzatore.zero_grad()

        output = modello(X_batch)

        perdita = criterio(output, y_batch)

        perdita.backward()

        ottimizzatore.step()

        perdita_totale += perdita.item() * len(X_batch)

    perdita_train = perdita_totale / len(train_dataset)

    modello.eval()

    with torch.no_grad():

        output_val = modello(X_val_tensor.to(device))

        perdita_val = criterio(
            output_val,
            y_val_tensor.to(device)
        ).item()

    print(
        f"Epoca {epoca + 1:3d} | "
        f"Loss train: {perdita_train:.4f} | "
        f"Loss val: {perdita_val:.4f}"
    )

Epoca   1 | Loss train: 0.7094 | Loss val: 0.6403
Epoca   2 | Loss train: 0.5441 | Loss val: 0.4713
Epoca   3 | Loss train: 0.4024 | Loss val: 0.3205
Epoca   4 | Loss train: 0.2502 | Loss val: 0.1822
Epoca   5 | Loss train: 0.1546 | Loss val: 0.1281
Epoca   6 | Loss train: 0.1187 | Loss val: 0.1083
Epoca   7 | Loss train: 0.0987 | Loss val: 0.0969
Epoca   8 | Loss train: 0.0868 | Loss val: 0.0911
Epoca   9 | Loss train: 0.0762 | Loss val: 0.0871
Epoca  10 | Loss train: 0.0689 | Loss val: 0.0835
Epoca  11 | Loss train: 0.0627 | Loss val: 0.0800
Epoca  12 | Loss train: 0.0574 | Loss val: 0.0785
Epoca  13 | Loss train: 0.0535 | Loss val: 0.0749
Epoca  14 | Loss train: 0.0496 | Loss val: 0.0733
Epoca  15 | Loss train: 0.0466 | Loss val: 0.0731
Epoca  16 | Loss train: 0.0430 | Loss val: 0.0734
Epoca  17 | Loss train: 0.0420 | Loss val: 0.0696
Epoca  18 | Loss train: 0.0387 | Loss val: 0.0672
Epoca  19 | Loss train: 0.0386 | Loss val: 0.0679
Epoca  20 | Loss train: 0.0349 | Loss val: 0.0655


In [34]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

modello.eval()

with torch.no_grad():

    output_test = modello(X_test_tensor.to(device))

    probabilita_test = torch.sigmoid(output_test)

    predizioni_test = (
        probabilita_test >= 0.5
    ).int()

y_test_reale = y_test_tensor.numpy().ravel()

predizioni_test = predizioni_test.cpu().numpy().ravel()

probabilita_test = probabilita_test.cpu().numpy().ravel()

print("Confusion matrix:")
print(confusion_matrix(y_test_reale, predizioni_test))

print("\nClassification report:")
print(
    classification_report(
        y_test_reale,
        predizioni_test,
        target_names=["No", "Yes"]
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(y_test_reale, probabilita_test)
)

Confusion matrix:
[[483   6]
 [  5 176]]

Classification report:
              precision    recall  f1-score   support

          No       0.99      0.99      0.99       489
         Yes       0.97      0.97      0.97       181

    accuracy                           0.98       670
   macro avg       0.98      0.98      0.98       670
weighted avg       0.98      0.98      0.98       670

ROC-AUC: 0.9986555039600493


In [41]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    accuracy_score
)

# ==================================================
# 1. PREPARAZIONE DEI DATI
# ==================================================

df_model = df.copy()

# Rimuovo la variabile correlata alla depressione
df_model = df_model.drop(columns=["Health_condition"])

# Separazione delle variabili indipendenti dal target
X = df_model.drop(columns=["Depression"])
y = df_model["Depression"].map({"No": 0, "Yes": 1})

# Controllo dei valori del target
if y.isna().any():
    raise ValueError("Il target contiene valori diversi da Yes e No.")

print("Dimensioni dataset:", df_model.shape)
print("\nDistribuzione target:")
print(df_model["Depression"].value_counts())


# ==================================================
# 2. SUDDIVISIONE TRAIN / VALIDATION / TEST
# ==================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nDimensioni dei dataset:")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


# ==================================================
# 3. PREPROCESSING
# ==================================================

colonne_numeriche = X.select_dtypes(
    include=["number"]
).columns.tolist()

colonne_categoriche = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

preprocessing = ColumnTransformer(
    transformers=[
        ("numeriche", StandardScaler(), colonne_numeriche),
        (
            "categoriche",
            OneHotEncoder(handle_unknown="ignore"),
            colonne_categoriche
        )
    ]
)

# Fit esclusivamente sul training set
X_train_array = preprocessing.fit_transform(X_train)
X_val_array = preprocessing.transform(X_val)
X_test_array = preprocessing.transform(X_test)


# ==================================================
# 4. CONVERSIONE IN TENSORI PYTORCH
# ==================================================

def converti_tensor(dati):
    if hasattr(dati, "toarray"):
        dati = dati.toarray()

    return torch.tensor(
        np.asarray(dati, dtype=np.float32),
        dtype=torch.float32
    )

X_train_tensor = converti_tensor(X_train_array)
X_val_tensor = converti_tensor(X_val_array)
X_test_tensor = converti_tensor(X_test_array)

y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

y_val_tensor = torch.tensor(
    y_val.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)


# ==================================================
# 5. DEVICE E DATALOADER
# ==================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDispositivo utilizzato:", device)

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)


# ==================================================
# 6. DEFINIZIONE DELLA RETE NEURALE
# ==================================================

class ReteNeurale(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.rete = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.rete(x)

input_size = X_train_tensor.shape[1]

modello = ReteNeurale(input_size).to(device)

numero_no = (y_train == 0).sum()
numero_yes = (y_train == 1).sum()

pos_weight = torch.tensor(
    [numero_no / numero_yes],
    dtype=torch.float32
).to(device)

criterio = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

ottimizzatore = torch.optim.Adam(
    modello.parameters(),
    lr=0.001
)


# ==================================================
# 7. ADDESTRAMENTO
# ==================================================

numero_epoche = 100

migliore_loss_val = float("inf")
pesi_migliori = None
pazienza = 20
epoche_senza_miglioramento = 0

for epoca in range(numero_epoche):

    modello.train()
    perdita_totale = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        ottimizzatore.zero_grad()

        output = modello(X_batch)

        perdita = criterio(output, y_batch)

        perdita.backward()
        ottimizzatore.step()

        perdita_totale += perdita.item() * len(X_batch)

    perdita_train = perdita_totale / len(train_dataset)

    # Validazione
    modello.eval()

    with torch.no_grad():

        output_val = modello(X_val_tensor.to(device))

        perdita_val = criterio(
            output_val,
            y_val_tensor.to(device)
        ).item()

    # Salvo i pesi migliori
    if perdita_val < migliore_loss_val:

        migliore_loss_val = perdita_val

        pesi_migliori = {
            nome: peso.detach().cpu().clone()
            for nome, peso in modello.state_dict().items()
        }

        epoche_senza_miglioramento = 0

    else:
        epoche_senza_miglioramento += 1

    if (epoca + 1) % 10 == 0 or epoca == 0:
        print(
            f"Epoca {epoca + 1:3d} | "
            f"Loss train: {perdita_train:.4f} | "
            f"Loss val: {perdita_val:.4f}"
        )

    # Early stopping
    if epoche_senza_miglioramento >= pazienza:
        print("\nEarly stopping all'epoca", epoca + 1)
        break


# Ripristino dei pesi migliori
modello.load_state_dict(pesi_migliori)
modello.eval()


# ==================================================
# 8. VALUTAZIONE FINALE SUL TEST SET
# ==================================================

with torch.no_grad():

    output_test = modello(X_test_tensor.to(device))

    probabilita_test = torch.sigmoid(output_test)

    predizioni_test = (
        probabilita_test >= 0.5
    ).int()

y_test_reale = y_test_tensor.numpy().ravel()

predizioni_test = (
    predizioni_test.cpu().numpy().ravel()
)

probabilita_test = (
    probabilita_test.cpu().numpy().ravel()
)

print("\n" + "=" * 50)
print("RISULTATI FINALI")
print("=" * 50)

print("\nConfusion matrix:")
print(confusion_matrix(y_test_reale, predizioni_test))

print("\nClassification report:")
print(
    classification_report(
        y_test_reale,
        predizioni_test,
        labels=[0, 1],
        target_names=["No", "Yes"],
        zero_division=0
    )
)

print(
    "Accuracy:",
    round(accuracy_score(y_test_reale, predizioni_test), 4)
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test_reale, probabilita_test), 4)
)


Dimensioni dataset: (4464, 12)

Distribuzione target:
Depression
No     3263
Yes    1201
Name: count, dtype: int64

Dimensioni dei dataset:
Training: (3124, 11)
Validation: (670, 11)
Test: (670, 11)

Dispositivo utilizzato: cuda
Epoca   1 | Loss train: 0.9567 | Loss val: 0.9118
Epoca  10 | Loss train: 0.7169 | Loss val: 1.0054
Epoca  20 | Loss train: 0.5462 | Loss val: 1.4102

Early stopping all'epoca 24

RISULTATI FINALI

Confusion matrix:
[[303 186]
 [ 64 117]]

Classification report:
              precision    recall  f1-score   support

          No       0.83      0.62      0.71       489
         Yes       0.39      0.65      0.48       181

    accuracy                           0.63       670
   macro avg       0.61      0.63      0.60       670
weighted avg       0.71      0.63      0.65       670

Accuracy: 0.6269
ROC-AUC: 0.6914


In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

modello_lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

modello_lr.fit(X_train_array, y_train)

probabilita_lr = modello_lr.predict_proba(X_val_array)[:, 1]

predizioni_lr = (probabilita_lr >= 0.5).astype(int)

print(classification_report(
    y_val,
    predizioni_lr,
    target_names=["No", "Yes"],
    zero_division=0
))

print("ROC-AUC:", roc_auc_score(y_val, probabilita_lr))

              precision    recall  f1-score   support

          No       0.85      0.60      0.70       490
         Yes       0.39      0.70      0.50       180

    accuracy                           0.63       670
   macro avg       0.62      0.65      0.60       670
weighted avg       0.72      0.63      0.65       670

ROC-AUC: 0.7049546485260771


In [ ]:

# ============================================================
# 1. IMPORTAZIONI
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ============================================================
# 2. PREPARAZIONE DEL DATASET
# ============================================================

df_model = df.copy()

# Escludo Health_condition
df_model = df_model.drop(columns=["Health_condition"])

# Variabili indipendenti
X = df_model.drop(columns=["Depression"])

# Target: No = 0, Yes = 1
y = df_model["Depression"].map({
    "No": 0,
    "Yes": 1
})

if y.isna().any():
    raise ValueError("Il target contiene valori diversi da No e Yes.")

print("Dimensioni dataset:", df_model.shape)

print("\nDistribuzione del target:")
print(df_model["Depression"].value_counts())


# ============================================================
# 3. SUDDIVISIONE TRAIN / VALIDATION / TEST
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nDimensioni dei dataset:")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


# ============================================================
# 4. PREPROCESSING
# ============================================================

colonne_numeriche = X.select_dtypes(
    include=["number"]
).columns.tolist()

colonne_categoriche = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

preprocessing = ColumnTransformer(
    transformers=[
        (
            "numeriche",
            StandardScaler(),
            colonne_numeriche
        ),
        (
            "categoriche",
            OneHotEncoder(handle_unknown="ignore"),
            colonne_categoriche
        )
    ]
)

# Il preprocessing viene adattato solo sul training set
X_train_array = preprocessing.fit_transform(X_train)

X_val_array = preprocessing.transform(X_val)

X_test_array = preprocessing.transform(X_test)


# ============================================================
# 5. CONVERSIONE IN TENSORI
# ============================================================

def converti_tensor(dati):

    if hasattr(dati, "toarray"):
        dati = dati.toarray()

    return torch.tensor(
        np.asarray(dati, dtype=np.float32),
        dtype=torch.float32
    )


X_train_tensor = converti_tensor(X_train_array)

X_val_tensor = converti_tensor(X_val_array)

X_test_tensor = converti_tensor(X_test_array)


y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

y_val_tensor = torch.tensor(
    y_val.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)


# ============================================================
# 6. DEVICE E DATALOADER
# ============================================================

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDispositivo:", device)

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)


# ============================================================
# 7. DEFINIZIONE DELLA RETE NEURALE
# ============================================================

class ReteNeurale(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.rete = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        return self.rete(x)


input_size = X_train_tensor.shape[1]

modello = ReteNeurale(input_size).to(device)

print("\nArchitettura della rete:")
print(modello)


# ============================================================
# 8. FUNZIONE DI PERDITA CON POS_WEIGHT
# ============================================================

numero_no = (y_train == 0).sum()

numero_yes = (y_train == 1).sum()

peso_positivo = numero_no / numero_yes

pos_weight = torch.tensor(
    [peso_positivo],
    dtype=torch.float32
).to(device)

print("\nPos weight:", round(float(peso_positivo), 3))

criterio = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

ottimizzatore = torch.optim.Adam(
    modello.parameters(),
    lr=0.001
)


# ============================================================
# 9. ADDESTRAMENTO CON EARLY STOPPING
# ============================================================

numero_epoche = 100
pazienza = 10

migliore_loss_val = float("inf")
pesi_migliori = None
epoche_senza_miglioramento = 0

for epoca in range(numero_epoche):

    modello.train()

    perdita_totale = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        ottimizzatore.zero_grad()

        output = modello(X_batch)

        perdita = criterio(output, y_batch)

        perdita.backward()

        ottimizzatore.step()

        perdita_totale += perdita.item() * len(X_batch)

    perdita_train = perdita_totale / len(train_dataset)

    # Validazione
    modello.eval()

    with torch.no_grad():

        output_val = modello(X_val_tensor.to(device))

        perdita_val = criterio(
            output_val,
            y_val_tensor.to(device)
        ).item()

    # Salvataggio dei pesi migliori
    if perdita_val < migliore_loss_val:

        migliore_loss_val = perdita_val

        pesi_migliori = {
            nome: peso.detach().cpu().clone()
            for nome, peso in modello.state_dict().items()
        }

        epoche_senza_miglioramento = 0

    else:

        epoche_senza_miglioramento += 1

    if (epoca + 1) % 10 == 0 or epoca == 0:

        print(
            f"Epoca {epoca + 1:3d} | "
            f"Loss train: {perdita_train:.4f} | "
            f"Loss val: {perdita_val:.4f}"
        )

    if epoche_senza_miglioramento >= pazienza:

        print("\nEarly stopping all'epoca", epoca + 1)

        break


# Ripristino dei pesi migliori
modello.load_state_dict(pesi_migliori)

modello.eval()


# ============================================================
# 10. PROBABILITÀ SUL VALIDATION SET
# ============================================================

with torch.no_grad():

    output_val = modello(X_val_tensor.to(device))

    probabilita_val = torch.sigmoid(
        output_val
    ).cpu().numpy().ravel()


y_val_reale = y_val.to_numpy()


# ============================================================
# 11. CONFRONTO DELLE SOGLIE SUL VALIDATION SET
# ============================================================

soglie = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60
]

risultati = []

print("\n" + "=" * 75)
print("CONFRONTO SOGLIE - VALIDATION SET")
print("=" * 75)

for soglia in soglie:

    predizioni = (
        probabilita_val >= soglia
    ).astype(int)

    precision = precision_score(
        y_val_reale,
        predizioni,
        zero_division=0
    )

    recall = recall_score(
        y_val_reale,
        predizioni,
        zero_division=0
    )

    f1 = f1_score(
        y_val_reale,
        predizioni,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_val_reale,
        predizioni
    )

    risultati.append({
        "Soglia": soglia,
        "Accuracy": accuracy,
        "Precision_Yes": precision,
        "Recall_Yes": recall,
        "F1_Yes": f1
    })


risultati_df = pd.DataFrame(risultati)

print(
    risultati_df.round(3).to_string(index=False)
)


# ============================================================
# 12. SCELTA DELLA SOGLIA
# ============================================================

# Esempio: seleziono la soglia con F1-score migliore
indice_migliore = risultati_df["F1_Yes"].idxmax()

soglia_migliore = risultati_df.loc[
    indice_migliore,
    "Soglia"
]

print("\nSoglia con F1-score migliore sul validation set:")
print(soglia_migliore)

print("\nMetriche corrispondenti:")
print(
    risultati_df.loc[indice_migliore].round(3)
)


# ============================================================
# 13. VALUTAZIONE FINALE SUL TEST SET
# ============================================================

# Il test set viene utilizzato solo dopo la scelta della soglia
with torch.no_grad():

    output_test = modello(X_test_tensor.to(device))

    probabilita_test = torch.sigmoid(
        output_test
    ).cpu().numpy().ravel()


y_test_reale = y_test.to_numpy()

predizioni_test = (
    probabilita_test >= soglia_migliore
).astype(int)


print("\n" + "=" * 75)
print("RISULTATI FINALI - TEST SET")
print("=" * 75)

print("\nSoglia utilizzata:", 0.2)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_test_reale,
        predizioni_test,
        labels=[0, 1]
    )
)

print("\nClassification report:")
print(
    classification_report(
        y_test_reale,
        predizioni_test,
        labels=[0, 1],
        target_names=["No", "Yes"],
        zero_division=0
    )
)

print(
    "Accuracy:",
    round(
        accuracy_score(y_test_reale, predizioni_test),
        4
    )
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(y_test_reale, probabilita_test),
        4
    )
)


Dimensioni dataset: (4464, 12)

Distribuzione del target:
Depression
No     3263
Yes    1201
Name: count, dtype: int64

Dimensioni dei dataset:
Training: (3124, 11)
Validation: (670, 11)
Test: (670, 11)

Dispositivo: cuda

Architettura della rete:
ReteNeurale(
  (rete): Sequential(
    (0): Linear(in_features=27, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

Pos weight: 2.719
Epoca   1 | Loss train: 0.9737 | Loss val: 0.9371
Epoca  10 | Loss train: 0.7799 | Loss val: 0.9078

Early stopping all'epoca 15

CONFRONTO SOGLIE - VALIDATION SET
 Soglia  Accuracy  Precision_Yes  Recall_Yes  F1_Yes
   0.20     0.425          0.313       0.956   0.472
   0.25     0.452          0.322       0.939   0.479
   0.30     0.478          0.329       0.911   0.484
   0.35     0.504          0.338       0.883   0.489
   0.40     0.545          0.356       0.856   0.5